<div dir="rtl">

# تعریف مسئله و متغیر هدف

در این بخش، هدف اصلی مشخص کردن دقیق مسئله‌ی پیش‌بینی است. از آن‌جا که مجموعه‌داده شامل آگهی‌های مختلف فروش، رهن و اجاره است، لازم است پیش از آموزش مدل تعیین شود که مدل قرار است دقیقاً چه مقدار مالی‌ای را پیش‌بینی کند.

برای این منظور، ابتدا انواع معامله‌های موجود در داده بررسی می‌شوند و سپس برای هر نوع معامله، ستون مناسب به‌عنوان متغیر هدف (`Target`) انتخاب خواهد شد.

مواردی که در این مرحله بررسی می‌شوند شامل موارد زیر هستند:

1. تعیین نوع مسئله‌ی پیش‌بینی برای آگهی‌های فروش و اجاره.
2. مشخص کردن ستون مناسب قیمت برای آگهی‌های فروش.
3. مشخص کردن مقدار مالی مناسب برای آگهی‌های رهن و اجاره.
4. بررسی امکان استفاده از یک مدل مشترک یا مدل‌های جداگانه برای انواع معامله.
5. بررسی تعداد داده‌های دارای Target معتبر برای هر گروه.
6. مشخص کردن داده‌هایی که به دلیل نداشتن Target معتبر، قابل استفاده در مدل‌سازی نیستند.

هدف این مرحله، رسیدن به یک تعریف شفاف و قابل دفاع از متغیر هدف است تا در مراحل بعدی، انتخاب Featureها، تقسیم داده و آموزش مدل بر اساس همین تعریف انجام شود.

</div>

In [2]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_parquet("../data/Divar_cleaned.parquet")


In [7]:
valid_transaction_types = [
    "residential-sell",
    "commercial-sell",
    "residential-rent",
    "commercial-rent"
]

model_df = df[df["cat2_slug"].isin(valid_transaction_types)].copy()

print("Rows before filtering:", len(df))
print("Rows after filtering:", len(model_df))

print("\nSelected transaction types:")
print(model_df["cat2_slug"].value_counts())

Rows before filtering: 999997
Rows after filtering: 950691

Selected transaction types:
cat2_slug
residential-sell    558707
residential-rent    276558
commercial-rent      76565
commercial-sell      38861
Name: count, dtype: int64[pyarrow]


<div dir="rtl">

### بررسی آگهی‌های اجاره کوتاه‌مدت

در این مرحله، آگهی‌های دسته‌ی `temporary-rent` به‌صورت جداگانه بررسی شدند تا مشخص شود آیا می‌توان برای آن‌ها نیز متغیر هدف مشترک `unified_price` را تعریف کرد یا خیر.

تعداد کل آگهی‌های اجاره کوتاه‌مدت برابر با **29,903** ردیف بود.

بررسی ستون‌های مالی نشان داد:

- تنها **5** ردیف دارای `price_value` بودند.
- تنها **3** ردیف دارای `credit_value` بودند.
- تنها **3** ردیف دارای `rent_value` بودند.
- ستون‌های `transformable_credit`، `transformed_credit`، `transformable_rent` و `transformed_rent` برای تمام این دسته فاقد مقدار بودند.
- ستون `rent_type` نیز تنها برای **2** ردیف مقدار داشت.
- `inferred_full_credit` نیز فقط برای **2** ردیف قابل تشخیص بود.

بنابراین تقریباً تمام آگهی‌های `temporary-rent` فاقد اطلاعات مالی لازم برای ساخت `unified_price` هستند.

در نتیجه، با وجود اینکه اجاره کوتاه‌مدت از نظر مفهومی جزو معاملات اجاره محسوب می‌شود، در این دیتاست امکان تعریف یک Target مالی قابل اتکا برای آن وجود ندارد. به همین دلیل، این دسته از داده‌های مورد استفاده برای مدل پیش‌بینی حذف می‌شود.

همچنین دسته‌ی `real-estate-services` نیز به دلیل اینکه مربوط به خدمات حوزه املاک است و نه معامله مستقیم یک ملک، وارد مدل نخواهد شد.

در نتیجه، انواع معامله‌ی مورد استفاده در مدل عبارت‌اند از:

- `residential-sell`
- `commercial-sell`
- `residential-rent`
- `commercial-rent`

</div>

In [8]:

is_sell = model_df["cat2_slug"].str.contains("sell", na=False)

is_full_credit = (
    (model_df["inferred_full_credit"] == True)
    .fillna(False)
)

conditions = [
    is_sell,
    is_full_credit
]

choices = [
    model_df["price_value"],
    model_df["transformable_credit"]
]

model_df["unified_price"] = np.select(
    conditions,
    choices,
    default=model_df["transformed_credit"]
)

# Zero cannot be used as a valid target
model_df["unified_price"] = model_df["unified_price"].replace(0, np.nan)

In [9]:
valid_target = model_df["unified_price"].notna()

print("Total modeling rows:", len(model_df))
print("Valid targets:", valid_target.sum())
print("Missing targets:", (~valid_target).sum())
print(
    "Target coverage:",
    round(valid_target.mean() * 100, 2),
    "%"
)

print("\nValid targets by transaction type:")
print(
    model_df.loc[valid_target, "cat2_slug"]
    .value_counts()
)

Total modeling rows: 950691
Valid targets: 680999
Missing targets: 269692
Target coverage: 71.63 %

Valid targets by transaction type:
cat2_slug
residential-sell    533260
residential-rent    104051
commercial-sell      31929
commercial-rent      11759
Name: count, dtype: int64[pyarrow]


<div dir="rtl">

### تعریف نهایی مسئله و متغیر هدف

در این پروژه یک مدل رگرسیون مشترک برای پیش‌بینی ارزش مالی آگهی‌های املاک در نظر گرفته شد.

چهار نوع معامله در داده‌های مدل نگه داشته شدند:

- فروش مسکونی
- فروش تجاری
- اجاره مسکونی
- اجاره تجاری

دسته‌ی `real-estate-services` به دلیل مربوط بودن به خدمات املاک و دسته‌ی `temporary-rent` به دلیل نبود اطلاعات مالی کافی برای تعریف متغیر هدف، از داده‌های مدل حذف شدند.

متغیر هدف مدل با نام `unified_price` تعریف شد تا انواع مختلف معاملات دارای یک Target عددی مشترک باشند.

منطق ساخت متغیر هدف به شکل زیر است:

- در آگهی‌های فروش از `price_value` استفاده می‌شود.
- در آگهی‌های رهن کامل، که با `inferred_full_credit` تشخیص داده می‌شوند، از `transformable_credit` استفاده می‌شود.
- در سایر آگهی‌های اجاره از `transformed_credit` استفاده می‌شود.
- مقادیر صفر به‌عنوان Target معتبر در نظر گرفته نمی‌شوند.

همچنین نوع معامله (`cat2_slug`) در مراحل بعدی به‌عنوان یکی از ویژگی‌های ورودی مدل استفاده خواهد شد تا مدل بتواند تفاوت میان انواع معاملات را تشخیص دهد.

</div>